# Steam Sales Dataset Analysis (LLM Batch Preprocessing)

This project focuses on LLM-based batch preprocessing to clean, standardize, and enrich large-scale Steam marketplace data before analysis. The goal is to transform raw, inconsistent data into a structured and analysis-ready format using Large Language Models (LLMs), enabling more accurate and meaningful downstream insights.

Batch processing is a data processing approach where large volumes of data are handled in groups (batches) rather than processing each record individually in real-time.

It is especially useful for large datasets where:

- Real-time processing is unnecessary
- Efficiency and scalability are priorities
- Data can be processed asynchronously

LLM Batch Preprocessing Workflow:

- Data Chunking - The dataset is split into smaller, manageable batches.
- LLM Invocation - Send each batch to the LLM with clear prompt instructions for preprocessing.
- Data Cleaning - Fix missing values, remove noise, and standardize formats.
- Data Enrichment -Convert unstructured data into structured form and add useful features.
- Validation & Merging - Check consistency and combine all processed batches into the final dataset.

## Import Libraries

In [49]:
import os
import requests 
import subprocess 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from litellm import completion
from pathlib import Path
import json
from sales_util import item_parser, item_preprocess
from sales_util.items_data import Item

In [2]:
# Load env file
load_dotenv(override=True)

True

In [55]:
BASE_DIR = Path(os.getenv("PROJECT_ROOT"))

## Load Dataset From HuggingFace

In [3]:
LITE_MODE = True

In [4]:
username = "KumudithaSilva"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

Loaded 22,000 items


In [7]:
for index, item in enumerate(items):
    item.id = index

## Ollama Initialization

In [16]:
subprocess.Popen("ollama serve", shell=True)

<Popen: returncode: None args: 'ollama serve'>

In [17]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [18]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME                ID              SIZE      MODIFIED     
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    2 months ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    2 months ago    



In [19]:
OLLAMA_API_URL = "http://localhost:11434/v1"

## LLM Batch Processing Prompt

In [8]:
SYSTEM_PROMPT = """Write a concise game description in 5–10 words.

Respond strictly in this format:
description: <short description>
"""

In [14]:
sample_item = items[2].small_description
sample_item

'Looking for an immersive VR physics puzzler? Chrono Weaver is a single-player co-op game. Work with copies of yourself as you travel through space and time, finding solutions to seemingly impossible tests. Get to work and get lost in this quirky sci-fi world.'

In [41]:
MODEL_OLLAMA = "ollama/llama3.2"
MODEL_OPEN_AI = "openai/gpt-4o-mini"

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sample_item}]
response = completion(messages=messages, model=MODEL_OLLAMA)
# response = completion(messages=messages, model=MODEL_OPEN_AI)

In [40]:
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

description: Solve VR physics puzzles as multiple selves throughout space-time.

Input tokens: 108
Output tokens: 15
Cost: 0.000 cents


##  JSONL Files

In [43]:
def make_jsonl(item):
    body = {"model": MODEL_OLLAMA, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.small_description}]}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [46]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "ollama/llama3.2", "messages": [{"role": "system", "content": "Write a concise game description in 5\\u201310 words.\\n\\nRespond strictly in this format:\\ndescription: <short description>\\n"}, {"role": "user", "content": "Downward Spear is a single player isometric tactical action game with stealth and horror elements. The ice caps have melted causing flooding of most coastal land, and released ancient viruses that consume their hosts."}]}}'

In [47]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [67]:
JSONL_PATH = BASE_DIR / "data"

In [69]:
make_file(0, 10, f"{JSONL_PATH}/0_10.jsonl")